In [1]:
from langchain_groq import ChatGroq
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import yfinance as yf
from langchain.agents import AgentType, initialize_agent
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool
from langchain_groq import ChatGroq

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from pymongo.operations import SearchIndexModel
from langchain_community.vectorstores import AtlasDB
# from langchain_mongodb import MongoDBAtlasVectorSearch
# from pymongo import MongoClient
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser
import PyPDF2
import pandas as pd
from langchain_core.documents import Document
from typing import Iterable
# from sentence_transformers import SentenceTransformer
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter
)
from langchain.text_splitter import NLTKTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables.utils import ConfigurableFieldSpec
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [2]:
"""
Stock Market Analysis Agent using LangGraph and Groq API

This code creates a langgraph agent that:
1. Takes user queries about stocks
2. Fetches real-time stock data from Alpha Vantage API
3. Analyzes the data using Groq's LLM API
4. Returns insights to the user

"""

import os
import json
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Any, Annotated, TypedDict, Sequence
from pydantic import BaseModel, Field

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

import langgraph
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolExecutor
from langgraph.prebuilt.tool_node import ToolNode

# Define API keys (in production, use environment variables)
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "your-groq-api-key")
ALPHAVANTAGE_API_KEY = os.environ.get("ALPHAVANTAGE_API_KEY", "your-alphavantage-api-key")

# Define State schema
class AgentState(TypedDict):
    """Represents the state of our agent."""
    messages: List[Any]
    tools_output: List[Dict[str, Any]]
    next: str

# Initialize the Groq Model
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",  # Using a capable model for financial analysis
    api_key=GROQ_API_KEY,
    temperature=0.1  # Lower temperature for more deterministic responses
)

# Define Stock Analysis Tools



In [5]:
from langchain.chains import SequentialChain
from mongodb_atlas import MongoDBAtlas, MongoDBAtlasDocumentManager
from langchain_core.runnables import RunnableLambda, RunnableConfig
from langchain_core.runnables import RunnableWithMessageHistory
class MongoDBAtlasQA:
    def __init__(self, mongo_uri, db_name, collection_name, embedding, index_name, llm, conversation_id: str= "test_session", user_id: str = "user1"):
        """Initialize the MongoDB Atlas QA system."""
        try:
            self.atlas= MongoDBAtlas(db_name, collection_name)
            self.document_manager= MongoDBAtlasDocumentManager(atlas=self.atlas)  
            self.atlas.create_vector_store(
                embedding=embedding,
                index_name=index_name,
                relevance_score_fn="cosine",
            )
            self._prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant that answers questions about stock market data."
                    ),
                    ("human", "{query}"),
                ]
            )
            self._history_prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant. Read the query and if it is vague look at the chat history to get more context else pass the query as it is. Output a consise query only do not answer the query."
                    ),
                    MessagesPlaceholder(variable_name="chat_history"),
                    ("human", "{query}"),
                ]
            )
            self.llm: ChatGroq=llm
            self.no_promopt_llm= llm | StrOutputParser()
            self.history_llm: ChatGroq = self._history_prompt | llm | StrOutputParser()
            # self.llm.bind_tools([self.get_similarity_search])
            self.llm= self._prompt| self.llm | StrOutputParser()
            
            self._config_fields = [
                ConfigurableFieldSpec(
                    id="user_id",
                    annotation=str,
                    name="User ID",
                    description="Unique identifier for a user.",
                    default="",
                    is_shared=True,
                ),
                ConfigurableFieldSpec(
                    id="conversation_id",
                    annotation=str,
                    name="Conversation ID",
                    description="Unique identifier for a conversation.",
                    default="",
                    is_shared=True,
                ),
            ]
            self.llm_with_history=RunnableWithMessageHistory(
                self.history_llm,
                self.get_chat_history,
                input_messages_key="query",
                history_messages_key="chat_history",
                history_factory_config=self._config_fields
            )
            self.config = {"configurable": {"user_id": "user1", "conversation_id": conversation_id}}
            summary_prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant that structures, formats and add details to answers based on the provided base knowledge and user query."
                    ),
                    ("human", "{text}"),
                ]
            )
            self.summary_llm = summary_prompt | self.no_promopt_llm
            self.user_id = user_id
            self.conversation_id = conversation_id
        except Exception as e:
            print(f"Error initializing MongoDBAtlasQA: {e}")

    def get_chat_history(self, user_id, conversation_id):
        try:
            return SQLChatMessageHistory(
                table_name="basic_stock_info_vector_search_chat_history",
                session_id=conversation_id,
                connection=os.getenv("POSTGRES_URI"),
            )
            
        except Exception as e:  
            print(f"Error retrieving chat history: {e}")
            return None

    def run_llm_with_history(self, text: str) -> str:
        """Run the LLM with chat history."""
        try:
            return self.llm_with_history
        except Exception as e:
            print(f"Error running LLM with history: {e}")
            return "Error processing your request."
        
    def run_query_mongo_react(self, text):
        """Query the knowledge base."""
        try:
            @tool(description="Perform a similarity search for info on stock market.")
            def get_similarity_search( text):
                """
                Perform a similarity search for info on stock market.
                
                Args:
                    text (str): The query string to search for.
                Returns:
                    List[Document]: A list of documents that match the query.
                """
                try:
                    results = self.atlas.similarity_search(text, k=5)
                    return results
                except Exception as e:
                    print(f"Error performing similarity search: {e}")
                    return None
                
            tools= [get_similarity_search]
            agent = initialize_agent(
                tools=tools,
                llm=self.no_promopt_llm,
                agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
                verbose=True
            )
            return agent.invoke({"query": text, "user_id": "user1", "conversation_id": "abc123"})
        
        except Exception as e:
            print(f"Error querying the knowledge base: {e}")
            return None
    
    def format_answer(self, text: str, query: str) -> str:
        """Summarize the answer."""
        try:
            text = f"Base Knowledge: {text}\nUser Query: {query}"
            summary_prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant that structures, formats and add details to answers based on the provided base knowledge and user query."
                    ),
                    ("human", "{text}"),
                ]
            )
            summary_llm = summary_prompt | self.no_promopt_llm
            return summary_llm
        except Exception as e:
            print(f"Error formatting the answer: {e}")
            return "Error summarizing the answer."
        
    def run(self, text: str)-> str:
        """Run the MongoDB Atlas QA system."""
        try:
            # Step 1: Refine query using chat history
            refine_query = RunnableWithMessageHistory(
                self.history_llm,
                self.get_chat_history,
                input_messages_key="query",
                history_messages_key="chat_history",
                history_factory_config=self._config_fields
            ).with_config(self.config)

            # Step 2: Similarity search as a lambda
            similarity_search = RunnableLambda(lambda refined_query: {
                "refined_query": refined_query,
                "documents": self.atlas.similarity_search(refined_query, k=2)
            })

            # Step 3: Format answer
            formatter = RunnableLambda(lambda inputs: {
                "text": f"Base Knowledge: {'\n\n'.join([doc.page_content for doc in inputs['documents']])}\nUser Query: {inputs['refined_query']}"
            }) | self.summary_llm

            # Final chain
            self.full_chain = refine_query | similarity_search | formatter
            result = self.full_chain.invoke({"query": text}, config=self.config)
            return result
        except Exception as e:
            print(f"Error running the MongoDB Atlas QA system: {e}")
            return "Error processing your request."
    

In [ ]:
from langchain.chains import SequentialChain
from mongodb_atlas import MongoDBAtlas, MongoDBAtlasDocumentManager
from langchain_core.runnables import RunnableLambda, RunnableConfig
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory

class LimitedSQLChatMessageHistory(SQLChatMessageHistory):
    """Custom SQLChatMessageHistory that limits the number of messages returned."""
    
    def __init__(self, *args, message_limit: int = 5, **kwargs):
        super().__init__(*args, **kwargs)
        self.message_limit = message_limit
    
    @property
    def messages(self):
        """Retrieve the last N messages from the chat history."""
        all_messages = super().messages
        return all_messages[-self.message_limit:] if len(all_messages) > self.message_limit else all_messages

class MongoDBAtlasQA:
    def __init__(self, mongo_uri, db_name, collection_name, embedding, index_name, llm, 
                 conversation_id: str = "test_session", user_id: str = "user1", 
                 history_limit: int = 10):
        """Initialize the MongoDB Atlas QA system."""
        try:
            self.atlas = MongoDBAtlas(db_name, collection_name)
            self.document_manager = MongoDBAtlasDocumentManager(atlas=self.atlas)  
            self.atlas.create_vector_store(
                embedding=embedding,
                index_name=index_name,
                relevance_score_fn="cosine",
            )
            
            # Store history limit
            self.history_limit = history_limit
            
            self._prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant that answers questions about stock market data."
                    ),
                    ("human", "{query}"),
                ]
            )
            self._history_prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant. Read the query and if it is vague look at the chat history to get more context else pass the query as it is. Output a consise query only do not answer the query."
                    ),
                    MessagesPlaceholder(variable_name="chat_history"),
                    ("human", "{query}"),
                ]
            )
            self.llm: ChatGroq = llm
            self.no_promopt_llm = llm | StrOutputParser()
            self.history_llm: ChatGroq = self._history_prompt | llm | StrOutputParser()
            self.llm = self._prompt | self.llm | StrOutputParser()
            
            self._config_fields = [
                ConfigurableFieldSpec(
                    id="user_id",
                    annotation=str,
                    name="User ID",
                    description="Unique identifier for a user.",
                    default="",
                    is_shared=True,
                ),
                ConfigurableFieldSpec(
                    id="conversation_id",
                    annotation=str,
                    name="Conversation ID",
                    description="Unique identifier for a conversation.",
                    default="",
                    is_shared=True,
                ),
            ]
            self.llm_with_history = RunnableWithMessageHistory(
                self.history_llm,
                self.get_chat_history,
                input_messages_key="query",
                history_messages_key="chat_history",
                history_factory_config=self._config_fields
            )
            self.config = {"configurable": {"user_id": "user1", "conversation_id": conversation_id}}
            summary_prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant that structures, formats and add details to answers based on the provided base knowledge and user query."
                    ),
                    ("human", "{text}"),
                ]
            )
            self.summary_llm = summary_prompt | self.no_promopt_llm
            self.user_id = user_id
            self.conversation_id = conversation_id
        except Exception as e:
            print(f"Error initializing MongoDBAtlasQA: {e}")

    def get_chat_history(self, user_id, conversation_id):
        """Get chat history with limited number of messages."""
        try:
            # Option 1: Use custom limited history class
            return LimitedSQLChatMessageHistory(
                table_name="basic_stock_info_vector_search_chat_history",
                session_id=conversation_id,
                connection=os.getenv("POSTGRES_URI"),
                message_limit=self.history_limit
            )
            
        except Exception as e:  
            print(f"Error retrieving chat history: {e}")
            return None

    def get_chat_history_alternative(self, user_id, conversation_id):
        """Alternative approach: manually limit messages after retrieval."""
        try:
            full_history = SQLChatMessageHistory(
                table_name="basic_stock_info_vector_search_chat_history",
                session_id=conversation_id,
                connection=os.getenv("POSTGRES_URI"),
            )
            
            # Get all messages and limit them
            all_messages = full_history.messages
            limited_messages = all_messages[-self.history_limit:] if len(all_messages) > self.history_limit else all_messages
            
            # Create a new history object with limited messages
            limited_history = SQLChatMessageHistory(
                table_name="basic_stock_info_vector_search_chat_history",
                session_id=f"{conversation_id}_limited",
                connection=os.getenv("POSTGRES_URI"),
            )
            
            # Clear and add limited messages
            limited_history.clear()
            for message in limited_messages:
                limited_history.add_message(message)
                
            return limited_history
            
        except Exception as e:  
            print(f"Error retrieving limited chat history: {e}")
            return None

    def run_llm_with_history(self, text: str) -> str:
        """Run the LLM with chat history."""
        try:
            return self.llm_with_history
        except Exception as e:
            print(f"Error running LLM with history: {e}")
            return "Error processing your request."
        
    def run_query_mongo_react(self, text):
        """Query the knowledge base."""
        try:
            @tool(description="Perform a similarity search for info on stock market.")
            def get_similarity_search(text):
                """
                Perform a similarity search for info on stock market.
                
                Args:
                    text (str): The query string to search for.
                Returns:
                    List[Document]: A list of documents that match the query.
                """
                try:
                    results = self.atlas.similarity_search(text, k=5)
                    return results
                except Exception as e:
                    print(f"Error performing similarity search: {e}")
                    return None
                
            tools = [get_similarity_search]
            agent = initialize_agent(
                tools=tools,
                llm=self.no_promopt_llm,
                agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
                verbose=True
            )
            return agent.invoke({"query": text, "user_id": "user1", "conversation_id": "abc123"})
        
        except Exception as e:
            print(f"Error querying the knowledge base: {e}")
            return None
    
    def format_answer(self, text: str, query: str) -> str:
        """Summarize the answer."""
        try:
            text = f"Base Knowledge: {text}\nUser Query: {query}"
            summary_prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant that structures, formats and add details to answers based on the provided base knowledge and user query."
                    ),
                    ("human", "{text}"),
                ]
            )
            summary_llm = summary_prompt | self.no_promopt_llm
            return summary_llm
        except Exception as e:
            print(f"Error formatting the answer: {e}")
            return "Error summarizing the answer."
        
    def run(self, text: str) -> str:
        """Run the MongoDB Atlas QA system."""
        try:
            # Step 1: Refine query using limited chat history
            refine_query = RunnableWithMessageHistory(
                self.history_llm,
                self.get_chat_history,
                input_messages_key="query",
                history_messages_key="chat_history",
                history_factory_config=self._config_fields
            ).with_config(self.config)

            # Step 2: Similarity search as a lambda
            similarity_search = RunnableLambda(lambda refined_query: {
                "refined_query": refined_query,
                "documents": self.atlas.similarity_search(refined_query, k=2)
            })

            # Step 3: Format answer
            formatter = RunnableLambda(lambda inputs: {
                "text": f"Base Knowledge: {'\n\n'.join([doc.page_content for doc in inputs['documents']])}\nUser Query: {inputs['refined_query']}"
            }) | self.summary_llm

            # Final chain
            self.full_chain = refine_query | similarity_search | formatter
            result = self.full_chain.invoke({"query": text}, config=self.config)
            return result
        except Exception as e:
            print(f"Error running the MongoDB Atlas QA system: {e}")
            return "Error processing your request."

# Usage example:
# qa_system = MongoDBAtlasQA(
#     mongo_uri="your_uri",
#     db_name="your_db", 
#     collection_name="your_collection",
#     embedding=your_embedding,
#     index_name="your_index",
#     llm=your_llm,
#     history_limit=5  # Only keep last 5 messages
# )

In [9]:
from langchain_groq import ChatGroq
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.runnables.utils import ConfigurableFieldSpec
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import SystemMessage
from langchain.agents import AgentType, initialize_agent

from mongodb_atlas import MongoDBAtlas, MongoDBAtlasDocumentManager


class LimitedSQLChatMessageHistory(SQLChatMessageHistory):
    """Custom SQLChatMessageHistory that limits the number of messages returned."""
    
    def __init__(self, *args, message_limit: int = 5, **kwargs):
        super().__init__(*args, **kwargs)
        self.message_limit = message_limit
    
    @property
    def messages(self):
        """Retrieve the last N messages from the chat history."""
        all_messages = super().messages
        return all_messages[-self.message_limit:] if len(all_messages) > self.message_limit else all_messages


class PromptManager:
    """Manages all prompt templates for the QA system."""
    
    @staticmethod
    def get_base_prompt():
        """Get the base system prompt template."""
        return ChatPromptTemplate.from_messages([
            SystemMessage(
                content="You are a helpful assistant that answers questions about stock market data."
            ),
            ("human", "{query}"),
        ])
    
    @staticmethod
    def get_history_prompt():
        """Get the history-aware prompt template."""
        return ChatPromptTemplate.from_messages([
            SystemMessage(
                content="You are a helpful assistant. Read the query and if it is vague look at the chat history to get more context else pass the query as it is. Output a consise query only do not answer the query."
            ),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{query}"),
        ])
    
    @staticmethod
    def get_summary_prompt():
        """Get the summary prompt template."""
        return ChatPromptTemplate.from_messages([
            SystemMessage(
                content="You are a helpful assistant that structures, formats and add details to answers based on the provided base knowledge and user query."
            ),
            ("human", "{text}"),
        ])


class LLMChainManager:
    """Manages LLM chains and their configurations."""
    
    def __init__(self, llm: ChatGroq):
        self.llm = llm
        self.prompt_manager = PromptManager()
        self._setup_chains()
    
    def _setup_chains(self):
        """Initialize all LLM chains."""
        # Base chains
        self.no_prompt_llm = self.llm | StrOutputParser()
        self.base_llm_chain = self.prompt_manager.get_base_prompt() | self.llm | StrOutputParser()
        self.history_llm_chain = self.prompt_manager.get_history_prompt() | self.llm | StrOutputParser()
        self.summary_llm_chain = self.prompt_manager.get_summary_prompt() | self.no_prompt_llm
    
    def get_base_chain(self):
        return self.base_llm_chain
    
    def get_history_chain(self):
        return self.history_llm_chain
    
    def get_summary_chain(self):
        return self.summary_llm_chain
    
    def get_no_prompt_chain(self):
        return self.no_prompt_llm


class ChatHistoryManager:
    """Manages chat history operations."""
    
    def __init__(self, history_limit: int = 2):
        self.history_limit = history_limit
        self.table_name = "basic_stock_info_vector_search_chat_history"
        self.postgres_uri = os.getenv("POSTGRES_URI")
    
    def get_limited_chat_history(self, user_id: str, conversation_id: str):
        """Get chat history with limited number of messages."""
        try:
            return LimitedSQLChatMessageHistory(
                table_name=self.table_name,
                session_id=conversation_id,
                connection=self.postgres_uri,
                message_limit=self.history_limit
            )
        except Exception as e:  
            print(f"Error retrieving chat history: {e}")
            return None

    def get_alternative_limited_history(self, user_id: str, conversation_id: str):
        """Alternative approach: manually limit messages after retrieval."""
        try:
            full_history = SQLChatMessageHistory(
                table_name=self.table_name,
                session_id=conversation_id,
                connection=self.postgres_uri,
            )
            
            # Get all messages and limit them
            all_messages = full_history.messages
            limited_messages = all_messages[-self.history_limit:] if len(all_messages) > self.history_limit else all_messages
            
            # Create a new history object with limited messages
            limited_history = SQLChatMessageHistory(
                table_name=self.table_name,
                session_id=f"{conversation_id}_limited",
                connection=self.postgres_uri,
            )
            
            # Clear and add limited messages
            limited_history.clear()
            for message in limited_messages:
                limited_history.add_message(message)
                
            return limited_history
            
        except Exception as e:  
            print(f"Error retrieving limited chat history: {e}")
            return None


class VectorSearchManager:
    """Manages vector search operations."""
    
    def __init__(self, atlas: MongoDBAtlas):
        self.atlas = atlas
    
    def create_search_tool(self):
        """Create the similarity search tool."""
        @tool(description="Perform a similarity search for info on stock market.")
        def get_similarity_search(text):
            """
            Perform a similarity search for info on stock market.
            
            Args:
                text (str): The query string to search for.
            Returns:
                List[Document]: A list of documents that match the query.
            """
            try:
                results = self.atlas.similarity_search(text, k=5)
                return results
            except Exception as e:
                print(f"Error performing similarity search: {e}")
                return None
        
        return get_similarity_search
    
    def similarity_search(self, query: str, k: int = 2):
        """Perform similarity search."""
        try:
            return self.atlas.similarity_search(query, k=k)
        except Exception as e:
            print(f"Error performing similarity search: {e}")
            return []


class AgentManager:
    """Manages agent operations."""
    
    def __init__(self, llm_chain, vector_search_manager: VectorSearchManager):
        self.llm_chain = llm_chain
        self.vector_search_manager = vector_search_manager
    
    def create_react_agent(self):
        """Create a ReAct agent with search tools."""
        tools = [self.vector_search_manager.create_search_tool()]
        agent = initialize_agent(
            tools=tools,
            llm=self.llm_chain,
            agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
            verbose=True
        )
        return agent


class ChainBuilder:
    """Builds complex chains for the QA system."""
    
    def __init__(self, llm_manager: LLMChainManager, history_manager: ChatHistoryManager, 
                 vector_manager: VectorSearchManager, config_fields: list):
        self.llm_manager = llm_manager
        self.history_manager = history_manager
        self.vector_manager = vector_manager
        self.config_fields = config_fields
    
    def build_history_aware_chain(self, config: dict):
        """Build a chain that uses chat history."""
        return RunnableWithMessageHistory(
            self.llm_manager.get_history_chain(),
            self.history_manager.get_limited_chat_history,
            input_messages_key="query",
            history_messages_key="chat_history",
            history_factory_config=self.config_fields
        ).with_config(config)
    
    def build_full_qa_chain(self, config: dict):
        """Build the complete QA chain."""
        # Step 1: Refine query using limited chat history
        refine_query = self.build_history_aware_chain(config)

        # Step 2: Similarity search as a lambda
        similarity_search = RunnableLambda(lambda refined_query: {
            "refined_query": refined_query,
            "documents": self.vector_manager.similarity_search(refined_query, k=2)
        })

        # Step 3: Format answer
        formatter = RunnableLambda(lambda inputs: {
            "text": f"Base Knowledge: {self._format_documents(inputs['documents'])}\nUser Query: {inputs['refined_query']}"
        }) | self.llm_manager.get_summary_chain()

        # Final chain
        return refine_query | similarity_search | formatter
    
    def _format_documents(self, documents):
        """Format documents for the prompt."""
        return '\n\n'.join([doc.page_content for doc in documents])


class MongoDBAtlasQA:
    """Main QA system that orchestrates all components."""
    
    def __init__(self, mongo_uri, db_name, collection_name, embedding, index_name, llm, 
                 conversation_id: str = "test_session", user_id: str = "user1", 
                 history_limit: int = 2):
        """Initialize the MongoDB Atlas QA system."""
        try:
            # Initialize core components
            self._setup_mongodb_atlas(db_name, collection_name, embedding, index_name)
            self._setup_managers(llm, history_limit)
            self._setup_configuration(user_id, conversation_id)
            self._setup_chains()
            
        except Exception as e:
            print(f"Error initializing MongoDBAtlasQA: {e}")

    def _setup_mongodb_atlas(self, db_name, collection_name, embedding, index_name):
        """Setup MongoDB Atlas components."""
        self.atlas = MongoDBAtlas(db_name, collection_name)
        self.document_manager = MongoDBAtlasDocumentManager(atlas=self.atlas)  
        self.atlas.create_vector_store(
            embedding=embedding,
            index_name=index_name,
            relevance_score_fn="cosine",
        )

    def _setup_managers(self, llm, history_limit):
        """Setup all manager components."""
        self.llm_manager = LLMChainManager(llm)
        self.history_manager = ChatHistoryManager(history_limit)
        self.vector_manager = VectorSearchManager(self.atlas)
        self.agent_manager = AgentManager(self.llm_manager.get_no_prompt_chain(), self.vector_manager)

    def _setup_configuration(self, user_id, conversation_id):
        """Setup configuration for the system."""
        self.user_id = user_id
        self.conversation_id = conversation_id
        
        self._config_fields = [
            ConfigurableFieldSpec(
                id="user_id",
                annotation=str,
                name="User ID",
                description="Unique identifier for a user.",
                default="",
                is_shared=True,
            ),
            ConfigurableFieldSpec(
                id="conversation_id",
                annotation=str,
                name="Conversation ID",
                description="Unique identifier for a conversation.",
                default="",
                is_shared=True,
            ),
        ]
        
        self.config = {"configurable": {"user_id": user_id, "conversation_id": conversation_id}}

    def _setup_chains(self):
        """Setup the chain builder and main chains."""
        self.chain_builder = ChainBuilder(
            self.llm_manager, 
            self.history_manager, 
            self.vector_manager, 
            self._config_fields
        )
        
        # Build the main QA chain
        self.full_chain = self.chain_builder.build_full_qa_chain(self.config)

    def run_llm_with_history(self, text: str) -> str:
        """Run the LLM with chat history."""
        try:
            history_chain = self.chain_builder.build_history_aware_chain(self.config)
            return history_chain.invoke({"query": text})
        except Exception as e:
            print(f"Error running LLM with history: {e}")
            return "Error processing your request."
        
    def run_query_mongo_react(self, text: str):
        """Query the knowledge base using ReAct agent."""
        try:
            agent = self.agent_manager.create_react_agent()
            return agent.invoke({"input": text})
        except Exception as e:
            print(f"Error querying the knowledge base: {e}")
            return None
    
    def format_answer(self, text: str, query: str) -> str:
        """Format and summarize the answer."""
        try:
            formatted_text = f"Base Knowledge: {text}\nUser Query: {query}"
            return self.llm_manager.get_summary_chain().invoke({"text": formatted_text})
        except Exception as e:
            print(f"Error formatting the answer: {e}")
            return "Error summarizing the answer."
        
    def run(self, text: str) -> str:
        """Run the MongoDB Atlas QA system."""
        try:
            result = self.full_chain.invoke({"query": text}, config=self.config)
            return result
        except Exception as e:
            print(f"Error running the MongoDB Atlas QA system: {e}")
            return "Error processing your request."


# Usage example:
# qa_system = MongoDBAtlasQA(
#     mongo_uri="your_uri",
#     db_name="your_db", 
#     collection_name="your_collection",
#     embedding=your_embedding,
#     index_name="your_index",
#     llm=your_llm,
#     history_limit=5  # Only keep last 5 messages
# )

In [11]:
qa_system = MongoDBAtlasQA(MONGO_URI, DB_NAME, COLLECTION_NAME, embedding, INDEX_NAME, llm, history_limit=5)
# qa_system.run_query_mongo_react("how to use stock market?")
qa_system.run(text="what is stock market?")

"**Definition and Purpose of the Stock Market**\n\nThe stock market, also known as the share market, is a platform where buyers and sellers come together to trade publicly listed shares of companies. The primary purpose of the stock market is to provide a regulated environment where companies can raise capital by issuing shares to the public, and investors can buy and sell these shares in a transparent and efficient manner.\n\n**Key Functions of the Stock Market:**\n\n1. **Capital Raising**: The stock market enables companies to raise funds by issuing shares to the public, which can be used to finance business operations, expand operations, or pay off debts.\n2. **Investment Opportunities**: The stock market provides investors with a wide range of investment opportunities, allowing them to buy and sell shares of publicly traded companies.\n3. **Liquidity**: The stock market provides liquidity to investors, allowing them to easily buy and sell shares at prevailing market prices.\n4. **P

In [6]:
import os
from langchain_groq import ChatGroq
MONGO_URI = os.getenv("MONGODB_ATLAS_CLUSTER_URI")
DB_NAME = "potato_trade"
COLLECTION_NAME = "vectorized_documents"
INDEX_NAME = "test_vector_index"

# llm = ChatGroq(model_name="llama3-8b-8192", api_key=os.getenv("GROQ_API_KEY"), async_client=True)
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",  # Using a capable model for financial analysis
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.1  # Lower temperature for more deterministic responses
)
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


/tmp/ipykernel_47821/1877144949.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/home/steeldev/Desktop/Github/PotatoTrade/mlenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-04 15:19:30.392237: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computa

In [5]:
qa_system = MongoDBAtlasQA(MONGO_URI, DB_NAME, COLLECTION_NAME, embedding, INDEX_NAME, llm)


In [91]:
qa_system.query("What is the current stock price of Apple?")



> Entering new AgentExecutor chain...
Error querying the knowledge base: Missing keys ['conversation_id', 'user_id'] in config['configurable'] Expected keys are ['conversation_id', 'user_id'].When using via .invoke() or .stream(), pass in a config; e.g., chain.invoke({'query': 'foo'}, {'configurable': {'conversation_id': '[your-value-here]', 'user_id': '[your-value-here]'}})


In [6]:
@tool(description="Perform a similarity search for info on stock market.")
def get_similarity_search_from_mongo(query: str):
    """Perform a similarity search for info on stock market."""
    try:
        print(f"Performing similarity search for query: {query}")
        qa_system = MongoDBAtlasQA(MONGO_URI, DB_NAME, COLLECTION_NAME, embedding, INDEX_NAME, llm)
        results = qa_system.get_similarity_search(query)
        if results is None:
            print("No results found.")
            return []
        
        return results
    except Exception as e:
        print(f"Error performing similarity search: {e}")
        return None

In [7]:
tools= [get_similarity_search_from_mongo]

In [8]:
from langgraph.prebuilt import ToolNode, tool_executor
tool_node = ToolNode(
    tools=tools,
    name="ToolExecutor",
)



In [9]:
from langchain_core.messages import AIMessage

# Creating an AI message object with single tool call
message_with_single_tool_call = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "get_similarity_search_from_mongo",
            "args": {"query": "what is stock market?"},
            "id": "tool_call_id",
            "type": "tool_call",
        }
    ],
)

# Invoke single tool call with created message
tool_node.invoke({"messages": [message_with_single_tool_call]})

Performing similarity search for query: what is stock market?


{'messages': [ToolMessage(content="[Document(metadata={'_id': '67e4cdb00e719b3b7e644dfe', 'doc_index': 0, 'chunk_index': 2}, page_content='The share market, also known as the stock market, is a platform where buyers and sellers come together to trade publicly listed shares of companies. The market is regulated by the Securities and Exchange Board of India (SEBI), which oversees the functioning of stock exchanges and ensures that listed companies comply with regulations and disclosure requirements.\\n\\nIf a company has issued 100 shares and you own 1 share, you own 1% stake in the company. The share market is where shares of different companies are traded.'), Document(metadata={'_id': '67e4cdb00e719b3b7e644dfc', 'doc_index': 0, 'chunk_index': 0}, page_content='Investing is a way to grow your money over time by putting it into various assets. One of the popular investment options is stocks. When you invest in stocks, you become a shareholder and can benefit from the company’s profits an

In [10]:
# llm.bind_tools(tools)
# Initialize the agent with the Groq model and tools
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)
agent.run("What is the stock market? Provide a detailed overview")

/tmp/ipykernel_15351/442757562.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(
/tmp/ipykernel_15351/442757562.py:9: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  agent.run("What is the stock market? Provide a detailed overview")




> Entering new AgentExecutor chain...
Thought: To provide a detailed overview of the stock market, I should start by performing a similarity search to gather relevant information on the topic. This will help me understand the basics of the stock market and provide a comprehensive answer.

Action: get_similarity_search_from_mongo
Action Input: "stock market overview"Performing similarity search for query: stock market overview

Observation: [Document(metadata={'_id': '67e568f11644c9727980c5ce', 'doc_index': 0, 'chunk_index': 4}, page_content='•4. It’s easy to learn how to profit from the stock market . \nBut You need to have your basics clear. Unless you do….you will be wasting \nyour time and loosing money. You need to be crystal clear of each and every \naspect of Investments , stock options, Stock Trading, Company, Shares , \nDividend & Types of Shares, Debentures, Securities,  Mutual Funds , IPO , \nFutures & Options, What does the Share Market consis t of? Exchanges, \nIndices, S

"The stock market is a platform where buyers and sellers trade publicly listed shares of companies, and it is regulated by the Securities and Exchange Board of India (SEBI). The market consists of various components, including stock exchanges, indices, and securities. Investors can benefit from the company's profits and growth by investing in stocks, but they also face the risk of losing their investment if the stock's value declines. The stock market can be volatile, and understanding its basics is essential for making informed decisions and potentially earning returns on investment. The benefits of investing in the stock market include the potential for long-term growth, liquidity, and diversification, while the risks include the potential for stock prices to decline, market volatility, and the risk of company-specific factors affecting stock performance. To navigate the stock market, investors should consider their investment objectives, risk appetite, and investment horizon, and an

In [125]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
memory=MemorySaver()
react_agent= create_react_agent(
    model=llm,
    tools=tools,
    checkpointer=memory
)

In [129]:
config={
    "configurable": {
        "user_id": "user1",
        "conversation_id": "conversation1",
        "thread_id": "thread1",
    }
}
from langchain_core.messages import HumanMessage
for chunk in react_agent.stream(
    {        "messages": [
        HumanMessage(content="What is the stock market? Provide a detailed overview")
    ]},
    config=config
):
    print(chunk)

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_vkf8', 'function': {'arguments': '{"query":"stock market overview"}', 'name': 'get_similarity_search_from_mongo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 848, 'prompt_tokens': 236, 'total_tokens': 1084, 'completion_time': 3.083636364, 'prompt_time': 0.089585336, 'queue_time': 0.06575490399999999, 'total_time': 3.1732217}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_6507bcfb6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-7c01ca4e-6737-4ab9-a543-f69f03600856-0', tool_calls=[{'name': 'get_similarity_search_from_mongo', 'args': {'query': 'stock market overview'}, 'id': 'call_vkf8', 'type': 'tool_call'}], usage_metadata={'input_tokens': 236, 'output_tokens': 848, 'total_tokens': 1084})]}}
Performing similarity search for query: stock market overview
{'tools': {'messages': [ToolMessage(content="[Document(metadata={'_id': '6

In [131]:
memory.storage

defaultdict(<function langgraph.checkpoint.memory.InMemorySaver.__init__.<locals>.<lambda>()>,
            {'thread1': defaultdict(dict,
                         {'': {'1f03a225-ab12-69ea-bfff-c999d59522f2': (('msgpack',
                             b'\x85\xa1v\x02\xa2ts\xd9 2025-05-26T11:12:47.334440+00:00\xa2id\xd9$1f03a225-ab12-69ea-bfff-c999d59522f2\xb0channel_versions\x81\xa9__start__\xd9300000000000000000000000000000001.0.3646421329242364\xadversions_seen\x81\xa9__input__\x80'),
                            ('msgpack',
                             b'\x87\xa6source\xa5input\xa6writes\x81\xa9__start__\xd95What is the stock market? Provide a detailed overview\xa7user_id\xa5user1\xafconversation_id\xadconversation1\xa9thread_id\xa7thread1\xa4step\xff\xa7parents\x80'),
                            None),
                           '1f03a226-fcc0-63b3-8000-627214135b01': (('msgpack',
                             b'\x85\xa1v\x02\xa2ts\xd9 2025-05-26T11:13:22.742550+00:00\xa2id\xd9$1f03a22

In [134]:
memory.list(config=config)

<generator object InMemorySaver.list at 0x77c172b91040>

In [2]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
# df = pd.read_json("hf://datasets/Josephgflowers/Finance-Instruct-500k/train.json", lines=True)
# save this data as jsonl file
# df.to_json("finance_instruct_500k.jsonl", orient="records", lines=True)
# Load the JSONL file into a DataFrame
df = pd.read_json("finance_instruct_500k.jsonl", lines=True)

In [3]:
df.head()

,system,user,assistant
0,\n,Explain tradeoffs between fiscal and monetary ...,Fiscal and monetary policy are the two main to...
1,\n,Explain the classical economic theory and its ...,The classical economic theory refers to the ec...
2,\n,Explain the difference between fiscal and mone...,Fiscal policy and monetary policy are the two ...
3,\n,Explain how central banks determine currency e...,1. Interest rates: By changing their benchmark...
4,\n,Explain how interest rates change with inflati...,"1. When inflation is rising, central banks typ..."


In [5]:
!pip install lamini -q
import lamini



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [30]:
formatted_responses[0]

{'input': "<|begin_of_text|><|start_header_id|>user<|end_header_id|>Explain tradeoffs between fiscal and monetary policy as tools in a nation's economic toolkit. Provide examples of past instances when each were utilized, the economic conditions that led to them being deployed, their intended effects, and an evaluation of their relative efficacy and consequences.<|eot_id|><|start_header_id|>assistant<|end_header_id|>",
 'output': 'Fiscal and monetary policy are the two main tools that governments have to influence economic activity. They each have benefits and drawbacks.\n\nFiscal policy refers to government spending and taxation decisions. Examples of fiscal policy include:\n\n• During the Great Recession, the U.S. government implemented a fiscal stimulus through the American Recovery and Reinvestment Act of 2009. This included increased spending on infrastructure, tax cuts, and expanded unemployment benefits. The intention was to boost aggregate demand and stimulate economic activity

In [18]:
df.columns

Index(['system', 'user', 'assistant'], dtype='object')

In [16]:
lamini_api_key= os.getenv("LAMINI_API_KEY")

llm = lamini.Lamini(model_name="meta-llama/Meta-Llama-3.1-8B-Instruct")
# formatted_responses = [
#   {
#     "input": f'<|begin_of_text|><|start_header_id|>user<|end_header_id|>{r["input"]}<|eot_id|><|start_header_id|>assistant<|end_header_id|>',
#     "output": f'{r["output"]}',
#   }
#   for r in df
# ]

In [22]:
import json

formatted_responses = []
for r in df.itertuples():
    formatted_responses.append({
        "input": f'<|begin_of_text|><|start_header_id|>user<|end_header_id|>{r.user}<|eot_id|><|start_header_id|>assistant<|end_header_id|>',
        "output": f'{r.assistant}',
    })

# Save the formatted responses to a JSONL file
with open("formatted_responses.jsonl", "w", encoding="utf-8") as f:
    for item in formatted_responses:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [24]:
# load the formatted responses from the JSONL file
with open("formatted_responses.jsonl", "r", encoding="utf-8") as f:
    formatted_responses = [json.loads(line) for line in f]
# Create a Lamini dataset

In [26]:
formatted_responses=formatted_responses[0:100]  # Display the first 5 formatted responses

In [28]:
from lamini import Lamini
llm=Lamini(model_name="meta-llama/Meta-Llama-3-8B-Instruct", api_key=lamini_api_key)

llm.tune(data_or_dataset_id=formatted_responses,
            finetune_args={'learning_rate':1.0e-4}
          )

Data pairs uploaded to local.

Your dataset id is: e0b0bf5bc131c56701ea81f22cc04126f95227069e800100506a215f56b5bdad . Consider using this in the future to train using the same data. 
Eg: llm.train(data_or_dataset_id='e0b0bf5bc131c56701ea81f22cc04126f95227069e800100506a215f56b5bdad')
Your job (18127) has been submitted! Track its status at your private link: https://api.lamini.ai/tune/18127


{'job_id': 18127,
 'status': 'CREATED',
 'dataset_id': 'e0b0bf5bc131c56701ea81f22cc04126f95227069e800100506a215f56b5bdad'}

In [ ]:
from langchain.chains import SequentialChain
from mongodb_atlas import MongoDBAtlas, MongoDBAtlasDocumentManager
from langchain_core.runnables import RunnableLambda, RunnableConfig
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory

class LimitedSQLChatMessageHistory(SQLChatMessageHistory):
    """Custom SQLChatMessageHistory that limits the number of messages returned."""
    
    def __init__(self, *args, message_limit: int = 5, **kwargs):
        super().__init__(*args, **kwargs)
        self.message_limit = message_limit
    
    @property
    def messages(self):
        """Retrieve the last N messages from the chat history."""
        all_messages = super().messages
        return all_messages[-self.message_limit:] if len(all_messages) > self.message_limit else all_messages

class MongoDBAtlasQA:
    def __init__(self, mongo_uri, db_name, collection_name, embedding, index_name, llm, 
                 conversation_id: str = "test_session", user_id: str = "user1", 
                 history_limit: int = 2):
        """Initialize the MongoDB Atlas QA system."""
        try:
            self.atlas = MongoDBAtlas(db_name, collection_name)
            self.document_manager = MongoDBAtlasDocumentManager(atlas=self.atlas)  
            self.atlas.create_vector_store(
                embedding=embedding,
                index_name=index_name,
                relevance_score_fn="cosine",
            )
            
            # Store history limit
            self.history_limit = history_limit
            
            self._prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant that answers questions about stock market data."
                    ),
                    ("human", "{query}"),
                ]
            )
            self._history_prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant. Read the query and if it is vague look at the chat history to get more context else pass the query as it is. Output a consise query only do not answer the query."
                    ),
                    MessagesPlaceholder(variable_name="chat_history"),
                    ("human", "{query}"),
                ]
            )
            self.llm: ChatGroq = llm
            self.no_promopt_llm = llm | StrOutputParser()
            self.history_llm: ChatGroq = self._history_prompt | llm | StrOutputParser()
            self.llm = self._prompt | self.llm | StrOutputParser()
            
            self._config_fields = [
                ConfigurableFieldSpec(
                    id="user_id",
                    annotation=str,
                    name="User ID",
                    description="Unique identifier for a user.",
                    default="",
                    is_shared=True,
                ),
                ConfigurableFieldSpec(
                    id="conversation_id",
                    annotation=str,
                    name="Conversation ID",
                    description="Unique identifier for a conversation.",
                    default="",
                    is_shared=True,
                ),
            ]
            self.llm_with_history = RunnableWithMessageHistory(
                self.history_llm,
                self.get_chat_history,
                input_messages_key="query",
                history_messages_key="chat_history",
                history_factory_config=self._config_fields
            )
            self.config = {"configurable": {"user_id": "user1", "conversation_id": conversation_id}}
            summary_prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant that structures, formats and add details to answers based on the provided base knowledge and user query."
                    ),
                    ("human", "{text}"),
                ]
            )
            self.summary_llm = summary_prompt | self.no_promopt_llm
            self.user_id = user_id
            self.conversation_id = conversation_id
        except Exception as e:
            print(f"Error initializing MongoDBAtlasQA: {e}")

    def get_chat_history(self, user_id, conversation_id):
        """Get chat history with limited number of messages."""
        try:
            # Option 1: Use custom limited history class
            return LimitedSQLChatMessageHistory(
                table_name="basic_stock_info_vector_search_chat_history",
                session_id=conversation_id,
                connection=os.getenv("POSTGRES_URI"),
                message_limit=self.history_limit
            )
            
        except Exception as e:  
            print(f"Error retrieving chat history: {e}")
            return None

    def get_chat_history_alternative(self, user_id, conversation_id):
        """Alternative approach: manually limit messages after retrieval."""
        try:
            full_history = SQLChatMessageHistory(
                table_name="basic_stock_info_vector_search_chat_history",
                session_id=conversation_id,
                connection=os.getenv("POSTGRES_URI"),
            )
            
            # Get all messages and limit them
            all_messages = full_history.messages
            limited_messages = all_messages[-self.history_limit:] if len(all_messages) > self.history_limit else all_messages
            
            # Create a new history object with limited messages
            limited_history = SQLChatMessageHistory(
                table_name="basic_stock_info_vector_search_chat_history",
                session_id=f"{conversation_id}_limited",
                connection=os.getenv("POSTGRES_URI"),
            )
            
            # Clear and add limited messages
            limited_history.clear()
            for message in limited_messages:
                limited_history.add_message(message)
                
            return limited_history
            
        except Exception as e:  
            print(f"Error retrieving limited chat history: {e}")
            return None

    def run_llm_with_history(self, text: str) -> str:
        """Run the LLM with chat history."""
        try:
            return self.llm_with_history
        except Exception as e:
            print(f"Error running LLM with history: {e}")
            return "Error processing your request."
        
    def run_query_mongo_react(self, text):
        """Query the knowledge base."""
        try:
            @tool(description="Perform a similarity search for info on stock market.")
            def get_similarity_search(text):
                """
                Perform a similarity search for info on stock market.
                
                Args:
                    text (str): The query string to search for.
                Returns:
                    List[Document]: A list of documents that match the query.
                """
                try:
                    results = self.atlas.similarity_search(text, k=5)
                    return results
                except Exception as e:
                    print(f"Error performing similarity search: {e}")
                    return None
                
            tools = [get_similarity_search]
            agent = initialize_agent(
                tools=tools,
                llm=self.no_promopt_llm,
                agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
                verbose=True
            )
            return agent.invoke({"query": text, "user_id": "user1", "conversation_id": "abc123"})
        
        except Exception as e:
            print(f"Error querying the knowledge base: {e}")
            return None
    
    def format_answer(self, text: str, query: str) -> str:
        """Summarize the answer."""
        try:
            text = f"Base Knowledge: {text}\nUser Query: {query}"
            summary_prompt = ChatPromptTemplate.from_messages(
                [
                    SystemMessage(
                        content="You are a helpful assistant that structures, formats and add details to answers based on the provided base knowledge and user query."
                    ),
                    ("human", "{text}"),
                ]
            )
            summary_llm = summary_prompt | self.no_promopt_llm
            return summary_llm
        except Exception as e:
            print(f"Error formatting the answer: {e}")
            return "Error summarizing the answer."
        
    def run(self, text: str) -> str:
        """Run the MongoDB Atlas QA system."""
        try:
            # Step 1: Refine query using limited chat history
            refine_query = RunnableWithMessageHistory(
                self.history_llm,
                self.get_chat_history,
                input_messages_key="query",
                history_messages_key="chat_history",
                history_factory_config=self._config_fields
            ).with_config(self.config)

            # Step 2: Similarity search as a lambda
            similarity_search = RunnableLambda(lambda refined_query: {
                "refined_query": refined_query,
                "documents": self.atlas.similarity_search(refined_query, k=2)
            })

            # Step 3: Format answer
            formatter = RunnableLambda(lambda inputs: {
                "text": f"Base Knowledge: {'\n\n'.join([doc.page_content for doc in inputs['documents']])}\nUser Query: {inputs['refined_query']}"
            }) | self.summary_llm

            # Final chain
            self.full_chain = refine_query | similarity_search | formatter
            result = self.full_chain.invoke({"query": text}, config=self.config)
            return result
        except Exception as e:
            print(f"Error running the MongoDB Atlas QA system: {e}")
            return "Error processing your request."

# Usage example:
# qa_system = MongoDBAtlasQA(
#     mongo_uri="your_uri",
#     db_name="your_db", 
#     collection_name="your_collection",
#     embedding=your_embedding,
#     index_name="your_index",
#     llm=your_llm,
#     history_limit=5  # Only keep last 5 messages
# )

"**Definition, Function, and Role of the Stock Market in Trading Securities**\n\nThe stock market, also known as the equity market, is a platform where companies raise capital by issuing shares of stock to the public, and investors buy and sell these shares in hopes of earning a profit. The stock market plays a vital role in facilitating the trading of securities, providing a mechanism for companies to raise capital, and allowing investors to participate in the growth and profits of businesses.\n\n**Definition:**\n\nThe stock market is a marketplace where securities, such as stocks, bonds, and mutual funds, are bought and sold. It is a platform that enables companies to raise capital by issuing shares of stock to the public, and provides a mechanism for investors to buy and sell these shares.\n\n**Function:**\n\nThe primary function of the stock market is to:\n\n1. **Facilitate the raising of capital**: Companies issue shares of stock to raise capital for various purposes, such as expa